# Simulation Study

In [1]:
from preamble import *

%config InlineBackend.figure_format = 'retina'


In [2]:
# dill.load_session("Sim_Lognormal_Gamma.pkl")

In [3]:
import sys

print("ABC version:", abc.__version__)
print("Python version:", sys.version)
print("Numpy version:", np.__version__)
print("PyMC3 version:", pm.__version__)
print("Arviz version:", arviz.__version__)

tic()

ABC version: 0.2.1
Python version: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0]
Numpy version: 1.22.3
PyMC3 version: 3.11.4
Arviz version: 0.12.1


In [4]:
FAST = False

# Processor information and SMC calibration parameters
if not FAST:
    numItersData = 11
    popSize = 1000
    popSizeModels = 1000
    epsMin = 0
else:
    numItersData = 6
    popSize = 500
    popSizeModels = 1000
    epsMin = 1

smcArgs = { "epsMin": epsMin, "verbose": True}
smcArgs["numProcs"] = 40

## Model selection

We generate individual claim sizes from three models 

- $\text{Weib}\left(k = 1/2,\beta =  e^{1/2}/ \Gamma(3/2)\right)$
- $\text{Gamma}(r = e^{1/2},m = 1)$
- $\text{LogNormal}(\mu = 0,\sigma = 1)$

Samples of sizes $25, 50$ are considered

In [5]:
from math import gamma

np.exp(1 / 2) / gamma(3 / 2)

1.8603827342052657

In [6]:
rg = default_rng(123)

sample_sizes = [25, 50, 75, 100, 150, 200]
T = sample_sizes[-1]

claim_data = pd.DataFrame(
    {
        "lognormal": abc.simulate_claim_sizes(rg, T, "lognormal", (0, 1)),
        "gamma": abc.simulate_claim_sizes(rg, T, "gamma", (np.exp(1 / 2), 1)),
        "weibull": abc.simulate_claim_sizes(
            rg, T, "weibull", (1 / 2, np.exp(1 / 2) / gamma(3 / 2))
        ),
    }
)

## ABC posterior for choosing between lognormal, Weibull and gamma to model the claim sizes

### Lognormal data

In [7]:
models_data = ["lognormal"]
models_fitted = ["gamma", "lognormal", "weibull"]

priorG = abc.IndependentUniformPrior([(0, 5), (0, 100)], ("r", "m"))
modelG = abc.Model(sev="gamma")

priorL = abc.IndependentUniformPrior([(-20, 20), (0, 5)], ("μ", "σ"))
modelL = abc.Model(sev="lognormal")

priorW = abc.IndependentUniformPrior([(1e-1, 5), (0, 100)], ("k", "δ"))
modelW = abc.Model(sev="weibull")
priors=[priorG,priorL,priorW]
models = [modelG, modelL, modelW]

In [8]:
model_proba_abc = pd.DataFrame(
    {"model_data": [], "model_fit": [], "ss": [], "model_probability_ABC": []}
)

# model_data = "lognormal"
for model_data in models_data:
    sevs = claim_data[model_data]
    for ss in sample_sizes:
        uData = sevs[:ss]
        %time fit = abc.smc(numItersData, popSizeModels, uData, models, priors=priors, **smcArgs)
        for k in range(len(models)):
            weights = fit.weights[fit.models == k]
            res_mp = pd.DataFrame(
                {
                    "model_data": pd.Series(model_data),
                    "model_fit": pd.Series([models_fitted[k]]),
                    "ss": np.array([ss]),
                    "model_probability_ABC": pd.Series(
                        np.sum(fit.weights[fit.models == k])
                    ),
                }
            )
            model_proba_abc = pd.concat([model_proba_abc, res_mp])
            model_proba_abc

Starting ABC-SMC with population size of 1000 and sample size of 25 (~> 25) on 40 processes.
Finished sampling from prior, eps = 51.71, time = 136.0s / 2.3m, popSize = 1000 -> 500, ESS = (327, 343, 330) -> (112, 193, 195), # sims = 1000, total # sims = 1000
	model populations = (327, 343, 330), model weights = [0.33 0.34 0.33]
Finished SMC iteration 1, eps = 1.70, time = 28.0s / 0.5m, popSize = 1564 -> 512, ESS = (306, 603, 606) -> (18, 445, 37), # sims = 2840, total # sims = 3840
	model populations = (322, 620, 622), model weights = [0.22 0.38 0.4 ]
Finished SMC iteration 2, eps = 1.69, time = 1.0s / 0.0m, popSize = 1529 -> 878, ESS = (40, 839, 211) -> (37, 254, 209), # sims = 3200, total # sims = 7040
	model populations = (50, 935, 544), model weights = [0.06 0.86 0.08]
Finished SMC iteration 3, eps = 0.99, time = 1.0s / 0.0m, popSize = 1889 -> 774, ESS = (55, 622, 719) -> (17, 59, 424), # sims = 3640, total # sims = 10680
	model populations = (84, 699, 1106), model weights = [0.16 0

## True posterior samples

We run a Bayesian analysis on the individual claim data and compuet the model probabilities when a Weibull or a gamma distribution is assumed. The prior distribution on the parameters are taken as independent uniform distribution (as in the ABC approach). 

### Fitting a Weibull and a gamma model to the individual loss data

In [9]:
Bayesian_Summary = pd.DataFrame(
    {
        "model_data": [],
        "model_fit": [],
        "ss": [],
        "param_1": [],
        "param_2": [],
        "marginal_log_likelihood": [],
    }
)

for model_data in models_data:
    sevs = claim_data[model_data]
    for model_fitted in models_fitted:

        for ss in sample_sizes:
            uData = sevs[:ss]
            print(
                f"Fitting a {model_fitted} model to {len(uData)} data points generated from a {model_data} model"
            )

            if model_fitted == "gamma":
                with pm.Model() as model_sev:
                    r = pm.Uniform("param_1", lower=0, upper=5)
                    m = pm.Uniform("param_2", lower=0, upper=100)
                    U = pm.Gamma("U", alpha=r, beta=1 / m, observed=uData)
                    %time trace = pm.sample_smc(popSize, random_seed=1, chains=1)

            elif model_fitted == "lognormal":
                with pm.Model() as model_sev:
                    μ = pm.Uniform("param_1", lower=-20, upper=20)
                    σ = pm.Uniform("param_2", lower=0, upper=5)
                    U = pm.Lognormal("U", mu=μ, sigma=σ, observed=uData)
                    %time trace = pm.sample_smc(popSize, random_seed=1, chains=1)

            elif model_fitted == "weibull":
                with pm.Model() as model_sev:
                    k = pm.Uniform("param_1", lower=1e-1, upper=5)
                    δ = pm.Uniform("param_2", lower=0, upper=100)
                    U = pm.Weibull("U", alpha=k, beta=δ, observed=uData)
                    %time trace = pm.sample_smc(popSize, random_seed=1, chains=1)

            ll = trace.report.log_marginal_likelihood[0]

            res = pd.DataFrame(
                {
                    "model_data": [model_data],
                    "model_fit": [model_fitted],
                    "ss": [ss],
                    "param_1": [trace["param_1"].mean()],
                    "param_2": [trace["param_2"].mean()],
                    "marginal_log_likelihood": [ll],
                }
            )
            Bayesian_Summary = pd.concat([Bayesian_Summary, res])

Fitting a gamma model to 25 data points generated from a lognormal model


Initializing SMC sampler...
Sampling 1 chain in 1 job
Stage:   0 Beta: 0.010
Stage:   1 Beta: 0.047
Stage:   2 Beta: 0.129
Stage:   3 Beta: 0.277
Stage:   4 Beta: 0.779
Stage:   5 Beta: 1.000


CPU times: user 2.49 s, sys: 694 ms, total: 3.19 s
Wall time: 31.8 s
Fitting a gamma model to 50 data points generated from a lognormal model


Initializing SMC sampler...
Sampling 1 chain in 1 job
Stage:   0 Beta: 0.005
Stage:   1 Beta: 0.024
Stage:   2 Beta: 0.068
Stage:   3 Beta: 0.153
Stage:   4 Beta: 0.433
Stage:   5 Beta: 1.000


CPU times: user 1.92 s, sys: 13.9 ms, total: 1.93 s
Wall time: 1.94 s
Fitting a gamma model to 75 data points generated from a lognormal model


Initializing SMC sampler...
Sampling 1 chain in 1 job
Stage:   0 Beta: 0.003
Stage:   1 Beta: 0.016
Stage:   2 Beta: 0.044
Stage:   3 Beta: 0.095
Stage:   4 Beta: 0.272
Stage:   5 Beta: 0.913
Stage:   6 Beta: 1.000


... (27 outputs left out)

Initializing SMC sampler...
Sampling 1 chain in 1 job
Stage:   0 Beta: 0.002
Stage:   1 Beta: 0.008
Stage:   2 Beta: 0.019
Stage:   3 Beta: 0.042
Stage:   4 Beta: 0.119
Stage:   5 Beta: 0.395
Stage:   6 Beta: 1.000


CPU times: user 2.3 s, sys: 7.95 ms, total: 2.31 s
Wall time: 2.32 s
Fitting a weibull model to 200 data points generated from a lognormal model


Initializing SMC sampler...
Sampling 1 chain in 1 job
Stage:   0 Beta: 0.001
Stage:   1 Beta: 0.006
Stage:   2 Beta: 0.014
Stage:   3 Beta: 0.031
Stage:   4 Beta: 0.085
Stage:   5 Beta: 0.280
Stage:   6 Beta: 0.961
Stage:   7 Beta: 1.000


CPU times: user 2.49 s, sys: 10.2 ms, total: 2.5 s
Wall time: 2.51 s


In [10]:
Bayesian_Summary

,model_data,model_fit,ss,param_1,param_2,marginal_log_likelihood
0,lognormal,gamma,25.0,1.452940,1.338451,-44.106862
0,lognormal,gamma,50.0,1.260282,1.540875,-86.467702
0,lognormal,gamma,75.0,1.325229,1.327695,-120.298522
0,lognormal,gamma,100.0,1.404371,1.180437,-152.491596
0,lognormal,gamma,150.0,1.350877,1.294970,-235.508674
0,lognormal,gamma,200.0,1.385706,1.198068,-301.205588
0,lognormal,lognormal,25.0,0.157524,1.017816,-44.897210
0,lognormal,lognormal,50.0,0.154357,1.021445,-85.473633
0,lognormal,lognormal,75.0,0.103541,0.958989,-117.065667
0,lognormal,lognormal,100.0,0.082713,0.910577,-147.395901


In [11]:
max_marginal_log_likelihood = (
    Bayesian_Summary[["model_data", "ss", "marginal_log_likelihood"]]
    .groupby(["model_data", "ss"])
    .max()
)
max_marginal_log_likelihood.reset_index(level=["model_data", "ss"], inplace=True)
max_marginal_log_likelihood.rename(
    columns={"marginal_log_likelihood": "max_marginal_log_likelihood"}
)

Bayesian_Summary_1 = pd.merge(
    Bayesian_Summary, max_marginal_log_likelihood, how="left", on=["model_data", "ss"]
)
Bayesian_Summary_1

Bayesian_Summary_1["BF"] = np.exp(
    Bayesian_Summary_1.marginal_log_likelihood_x
    - Bayesian_Summary_1.marginal_log_likelihood_y
)

Bayesian_Summary_1
sum_BF = (
    Bayesian_Summary_1[["ss", "model_data", "BF"]].groupby(["ss", "model_data"]).sum()
)
sum_BF.reset_index(level=["model_data", "ss"], inplace=True)

Bayesian_Summary_2 = pd.merge(
    Bayesian_Summary_1, sum_BF, how="left", on=["model_data", "ss"]
)
Bayesian_Summary_2
Bayesian_Summary_2["model_probability"] = (
    Bayesian_Summary_2.BF_x / Bayesian_Summary_2.BF_y
)
Bayesian_Summary_2

,model_data,model_fit,ss,param_1,param_2,marginal_log_likelihood_x,marginal_log_likelihood_y,BF_x,BF_y,model_probability
0,lognormal,gamma,25.0,1.452940,1.338451,-44.106862,-44.106862,1.000000e+00,2.297620,4.352330e-01
1,lognormal,gamma,50.0,1.260282,1.540875,-86.467702,-85.473633,3.700679e-01,1.539944,2.403126e-01
2,lognormal,gamma,75.0,1.325229,1.327695,-120.298522,-117.065667,3.944473e-02,1.050331,3.755457e-02
3,lognormal,gamma,100.0,1.404371,1.180437,-152.491596,-147.395901,6.123047e-03,1.007013,6.080408e-03
4,lognormal,gamma,150.0,1.350877,1.294970,-235.508674,-227.963629,5.287236e-04,1.000575,5.284198e-04
5,lognormal,gamma,200.0,1.385706,1.198068,-301.205588,-290.913671,3.390605e-05,1.000035,3.390486e-05
6,lognormal,lognormal,25.0,0.157524,1.017816,-44.897210,-44.106862,4.536870e-01,2.297620,1.974595e-01
7,lognormal,lognormal,50.0,0.154357,1.021445,-85.473633,-85.473633,1.000000e+00,1.539944,6.493743e-01
8,lognormal,lognormal,75.0,0.103541,0.958989,-117.065667,-117.065667,1.000000e+00,1.050331,9.520808e-01
9,lognormal,lognormal,100.0,0.082713,0.910577,-147.395901,-147.395901,1.000000e+00,1.007013,9.930363e-01


In [12]:
# # Frequency-Loss Model
# α, p, k, β = 4, 2 / 3, 1 / 3, 1
# rg = default_rng(123)
# uData_10000 = abc.simulate_claim_sizes(rg, 10000, sev, θ_sev)
# r_mle, m_mle, BIC = infer_gamma(uData_10000, [1, 1])

# θ_plot = [[α, p, k, β], [α, p, np.NaN, np.NaN]]
# θ_mle = [[np.NaN, np.NaN, np.NaN, np.NaN], [np.NaN, np.NaN, r_mle, m_mle]]

In [13]:
model_proba = pd.merge(
    Bayesian_Summary_2[["model_data", "model_fit", "ss", "model_probability"]],
    model_proba_abc,
    how="left",
    on=["model_data", "model_fit", "ss"],
).round(2)
model_proba

,model_data,model_fit,ss,model_probability,model_probability_ABC
0,lognormal,gamma,25.0,0.44,0.45
1,lognormal,gamma,50.0,0.24,0.43
2,lognormal,gamma,75.0,0.04,0.30
3,lognormal,gamma,100.0,0.01,0.24
4,lognormal,gamma,150.0,0.00,0.14
5,lognormal,gamma,200.0,0.00,0.05
6,lognormal,lognormal,25.0,0.20,0.18
7,lognormal,lognormal,50.0,0.65,0.34
8,lognormal,lognormal,75.0,0.95,0.53
9,lognormal,lognormal,100.0,0.99,0.67


In [14]:
print(
    pd.pivot_table(
        model_proba,
        values=["model_probability", "model_probability_ABC"],
        index=["ss"],
        columns=["model_fit"],
        aggfunc={"model_probability": np.mean, "model_probability_ABC": np.mean},
    ).to_latex()
)

\begin{tabular}{lrrrrrr}
\toprule
 & \multicolumn{3}{r}{model_probability} & \multicolumn{3}{r}{model_probability_ABC} \\
model_fit & gamma & lognormal & weibull & gamma & lognormal & weibull \\
ss &  &  &  &  &  &  \\
\midrule
25.000000 & 0.440000 & 0.200000 & 0.370000 & 0.450000 & 0.180000 & 0.370000 \\
50.000000 & 0.240000 & 0.650000 & 0.110000 & 0.430000 & 0.340000 & 0.230000 \\
75.000000 & 0.040000 & 0.950000 & 0.010000 & 0.300000 & 0.530000 & 0.180000 \\
100.000000 & 0.010000 & 0.990000 & 0.000000 & 0.240000 & 0.670000 & 0.090000 \\
150.000000 & 0.000000 & 1.000000 & 0.000000 & 0.140000 & 0.790000 & 0.070000 \\
200.000000 & 0.000000 & 1.000000 & 0.000000 & 0.050000 & 0.930000 & 0.020000 \\
\bottomrule
\end{tabular}



In [15]:
elapsed = toc()
print(f"Notebook time = {elapsed:.0f} secs = {elapsed/60:.2f} mins")

Notebook time = 468 secs = 7.80 mins


In [16]:
dill.dump_session("Sim_Lognormal_Gamma.pkl")